# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


In [17]:
import os

import torch
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops.boxes import masks_to_boxes
from torchvision.transforms.v2 import functional as F
from typing import List

import cv2
import numpy as np
import matplotlib.pyplot as plt


COLORS = [51, 102, 153, 204, 255]


class ColonySegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms):
        self.root = root
        self.transforms = transforms

        # load all image files, sorting them to ensure that they are aligned
        self.imgs = list(sorted(os.listdir(os.path.join(root, "images"))))
        self.masks = list(sorted(os.listdir(os.path.join(root, "class_masks"))))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.imgs[idx])
        mask_path = os.path.join(self.root, "class_masks", self.masks[idx])
        img = read_image(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

        masks = np.zeros((num_labels - 1, *mask.shape), dtype=np.uint8)
        class_labels = np.zeros(num_labels - 1, dtype=np.uint8)

        for i in range(1, num_labels):
            # extract the mask for the current connected component
            region_mask = labels == i

            masks[i - 1] = region_mask

            color = np.median(mask[region_mask])

            if color == COLORS[0]:
                class_labels[i - 1] = 1
            elif color == COLORS[1]:
                class_labels[i - 1] = 2
            elif color == COLORS[2]:
                class_labels[i - 1] = 3
            elif color == COLORS[3]:
                class_labels[i - 1] = 4
            elif color == COLORS[4]:
                class_labels[i - 1] = 5

        assert len(masks) == len(class_labels)

        masks = torch.tensor(masks, dtype=torch.uint8)

        boxes = masks_to_boxes(masks)

        num_masks = len(masks)

        img = tv_tensors.Image(img)

        target = {
            "boxes": tv_tensors.BoundingBoxes(
                boxes, format="XYXY", canvas_size=F.get_size(img)
            ),
            "masks": tv_tensors.Mask(masks),
            # there is only one class (colony)
            "labels": torch.tensor(class_labels, dtype=torch.int64),
            "image_id": idx,
            "iscrowd": torch.zeros(num_masks, dtype=torch.int64),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [2]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def get_model_instance_segmentation(num_classes):
    # load an instance segmentation model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn_v2(
        weights="DEFAULT", box_detections_per_image=500
    )

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model

In [3]:
from torchvision.transforms import v2 as T


def get_transforms(is_train: bool):
    """Get the transforms for the dataset.

    Args:
        is_train (bool): Whether the transforms are for training or testing.

    Returns:
        func: A function that takes in an image and target and applies the transforms.
    """

    transforms = []

    if is_train:
        transforms.append(T.RandomHorizontalFlip(0.5))
        # transforms.append(T.RandomVerticalFlip(0.5))
        # Rotation by 0, 90, 180, or 270 degrees
        # transforms.append(
        #     T.RandomChoice(
        #         [
        #             T.RandomRotation((0, 0)),
        #             T.RandomRotation((90, 90)),
        #             T.RandomRotation((180, 180)),
        #             T.RandomRotation((270, 270)),
        #         ]
        #     )
        # )
        # transforms.append(T.ColorJitter(brightness=0.1, contrast=0.1))

    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [18]:
import utils

model = get_model_instance_segmentation(len(COLORS) + 1)
dataset = ColonySegmentationDataset(
    "../../training_data", get_transforms(is_train=True)
)
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
153.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
153.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
51.0
Found 98 colonies in image 42
Class labels: 98
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
102.0
102.0
51.0
51.0
51.0
102.0
51.0
51.0
51.0
51.0
51.0
102.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
102.0
51.0
51.0
51.0
51.0
51.0
51.0
102.0
51.0
51.0
51.0
51.0
51.0
51.0
Found 71 colonies in image 101
Class labels: 71
{'loss_classifier': tensor(1.9006, grad_fn=<Nl

## Training and Validation


In [19]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device("cpu")

# Check for accelerator availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA for training.")
else:
    print("Using CPU for training.")

Using CUDA for training.


In [13]:
from engine import evaluate, train_one_epoch


# our dataset has two classes only - background and colony
num_classes = len(COLORS) + 1
# use our dataset and defined transformations
dataset = ColonySegmentationDataset(
    "../../training_data", get_transforms(is_train=True)
)
dataset_test = ColonySegmentationDataset(
    "../../training_data", get_transforms(is_train=False)
)

# split the dataset in train and test set (last 30 images are test set)
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-25])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-25:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=4, num_workers=4, shuffle=True, collate_fn=utils.collate_fn
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=utils.collate_fn,
)

model = get_model_instance_segmentation(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_test, device=device)

print("That's it!")

# Save the model to disk
torch.save(model.state_dict(), "model.pth")

Epoch: [0]  [ 0/22]  eta: 0:02:54  lr: 0.000243  loss: 5.1359 (5.1359)  loss_classifier: 0.7724 (0.7724)  loss_box_reg: 0.3839 (0.3839)  loss_mask: 1.6457 (1.6457)  loss_objectness: 1.7725 (1.7725)  loss_rpn_box_reg: 0.5614 (0.5614)  time: 7.9221  data: 7.0726  max mem: 10304
Epoch: [0]  [10/22]  eta: 0:00:33  lr: 0.002621  loss: 3.0071 (2.9030)  loss_classifier: 0.4826 (0.5194)  loss_box_reg: 0.4648 (0.4718)  loss_mask: 0.7083 (0.8507)  loss_objectness: 0.8030 (0.8570)  loss_rpn_box_reg: 0.1586 (0.2041)  time: 2.7866  data: 1.8644  max mem: 11577
Epoch: [0]  [20/22]  eta: 0:00:05  lr: 0.005000  loss: 1.3591 (2.0283)  loss_classifier: 0.2727 (0.3273)  loss_box_reg: 0.4425 (0.4268)  loss_mask: 0.3050 (0.5887)  loss_objectness: 0.1868 (0.5227)  loss_rpn_box_reg: 0.0963 (0.1627)  time: 2.2793  data: 1.3398  max mem: 11577
Epoch: [0]  [21/22]  eta: 0:00:02  lr: 0.005000  loss: 1.3320 (1.9760)  loss_classifier: 0.2033 (0.3145)  loss_box_reg: 0.4368 (0.4179)  loss_mask: 0.3050 (0.5779)  loss

KeyboardInterrupt: 

In [ ]:
import torch
import gradio as gr
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks

model = get_model_instance_segmentation(2).to(device)
model.load_state_dict(torch.load("model.pth", map_location=device))

eval_transform = get_transforms(is_train=False)


def predict(image):
    # convert numpy image to tensor
    image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

    model.eval()
    with torch.no_grad():
        print(type(image))
        x = eval_transform(image)
        print(type(x))
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    # Set the desired score threshold
    score_threshold = 0.9

    # Create a mask for scores above the threshold
    score_mask = torch.tensor(pred["scores"]) > score_threshold

    # Filter the labels and boxes using the mask
    filtered_labels = [label for label, keep in zip(pred["labels"], score_mask) if keep]
    filtered_boxes = pred["boxes"][score_mask]

    # Update the prediction dictionary with filtered values
    pred = {
        "labels": filtered_labels,
        "scores": pred["scores"][score_mask],
        "boxes": filtered_boxes,
        "masks": pred["masks"][score_mask],
    }

    pred_labels = [f"{score:.3f}" for score in pred["scores"]]

    pred_boxes = pred["boxes"].long()
    output_image = draw_bounding_boxes(
        image,
        pred_boxes,
        pred_labels,
        colors="red",
        width=3,
        font_size=75,
        font="Inter-Regular.ttf",
    )

    masks = (pred["masks"] > 0.7).squeeze(1)
    output_image = draw_segmentation_masks(
        output_image, masks, alpha=0.5, colors="yellow"
    )

    return output_image.permute(1, 2, 0).numpy(), len(pred["scores"])


demo = gr.Interface(
    fn=predict,
    inputs="image",
    outputs=["image", gr.Number(label="Number of Colonies detected")],
    title="Colony Detection",
    description="Detect colonies in images.",
)

demo.launch(inline=False)